# PySpark

## Initialize Session

- A Spark session needs to be initialized. 
- With the help of SparkSession, DataFrame can be created and registered as tables.

In [65]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PysparkFTDS")\
    .config("spark.sql.shuffle.partitions", "50")\
    .config("spark.driver.maxResultSize", "5g")\
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .getOrCreate()

### Stop Session

In [64]:
spark.stop()

## Create DataFrame

In [66]:
from datetime import datetime, date
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=date(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=date(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=date(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
])

df

DataFrame[a: bigint, b: double, c: string, d: date, e: timestamp]

### Create DataFrame with specified schema

In [67]:
from datetime import datetime, date
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=date(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=date(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=date(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
], schema='a int, b float, c string, d date, e timestamp')

df

DataFrame[a: int, b: float, c: string, d: date, e: timestamp]

### Create DataFrame from Pandas DataFrame

In [68]:
import pandas as pd

pandas_df = pd.read_csv('data.csv', index_col=0)
df = spark.createDataFrame(pandas_df)

df

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  PyArrow >= 4.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


DataFrame[car_id: bigint, symboling: bigint, carname: string, fueltype: string, aspiration: string, doornumber: string, carbody: string, drivewheel: string, enginelocation: string, wheelbase: double, carlength: double, carwidth: double, carheight: double, curbweight: double, enginetype: string, cylindernumber: string, enginesize: bigint, fuelsystem: string, boreratio: double, stroke: double, compressionratio: double, horsepower: bigint, peakrpm: bigint, citympg: bigint, highwaympg: bigint, price: double]

### Create DataFrame from RDD

In [94]:
rdd = spark.sparkContext.parallelize([
    (1, 2., 'string1', date(2000, 1, 1), datetime(2000, 1, 1, 12, 0)),
    (2, 3., 'string2', date(2000, 2, 1), datetime(2000, 1, 2, 12, 0)),
    (3, 4., 'string3', date(2000, 3, 1), datetime(2000, 1, 3, 12, 0))
])
df = spark.createDataFrame(rdd, schema=['a', 'b', 'c', 'd', 'e'])

df

DataFrame[a: bigint, b: double, c: string, d: date, e: timestamp]

### Create DataFrame from Files

```py
# JSON
dataframe = sc.read.json('file_name.json')

# TXT FILES
dataframe_txt = sc.read.text('file_name.txt')

# CSV FILES
dataframe_csv = sc.read.csv('file_name.csv')

# PARQUET FILES
dataframe_parquet = sc.read.load('file_name.parquet')

# Export to File
sc.write.csv('ftds.csv', header=True)
```

## View Data

### Show All Data

In [70]:
df.show()

+---+---+-------+----------+-------------------+
|  a|  b|      c|         d|                  e|
+---+---+-------+----------+-------------------+
|  1|2.0|string1|2000-01-01|2000-01-01 12:00:00|
|  2|3.0|string2|2000-02-01|2000-01-02 12:00:00|
|  3|4.0|string3|2000-03-01|2000-01-03 12:00:00|
+---+---+-------+----------+-------------------+



#### Show Top N

In [71]:
df.show(1)

+---+---+-------+----------+-------------------+
|  a|  b|      c|         d|                  e|
+---+---+-------+----------+-------------------+
|  1|2.0|string1|2000-01-01|2000-01-01 12:00:00|
+---+---+-------+----------+-------------------+
only showing top 1 row



In [72]:
# show top N record(s) vertically, this can also be applied to all rows
df.show(1, vertical=True)

-RECORD 0------------------
 a   | 1                   
 b   | 2.0                 
 c   | string1             
 d   | 2000-01-01          
 e   | 2000-01-01 12:00:00 
only showing top 1 row



### Show Schema

In [73]:
df.printSchema()

root
 |-- a: long (nullable = true)
 |-- b: double (nullable = true)
 |-- c: string (nullable = true)
 |-- d: date (nullable = true)
 |-- e: timestamp (nullable = true)



```py
# get all columns
df.columns
```

### Show Summary

In [75]:
df.describe()

DataFrame[summary: string, a: string, b: string, c: string]

In [76]:
# show summary along with descriptive stats for all numeric columns
df.describe().show()

+-------+---+---+-------+
|summary|  a|  b|      c|
+-------+---+---+-------+
|  count|  3|  3|      3|
|   mean|2.0|3.0|   NULL|
| stddev|1.0|1.0|   NULL|
|    min|  1|2.0|string1|
|    max|  3|4.0|string3|
+-------+---+---+-------+



### Output

In [77]:
# convert DataFrame into list of Rows
df.collect()

[Row(a=1, b=2.0, c='string1', d=datetime.date(2000, 1, 1), e=datetime.datetime(2000, 1, 1, 12, 0)),
 Row(a=2, b=3.0, c='string2', d=datetime.date(2000, 2, 1), e=datetime.datetime(2000, 1, 2, 12, 0)),
 Row(a=3, b=4.0, c='string3', d=datetime.date(2000, 3, 1), e=datetime.datetime(2000, 1, 3, 12, 0))]

**Note**: *This can throw an out-of-memory error when the dataset is too large to fit in the driver side because it collects all the data from executors to the driver side.*

```py
# take N data from head
df.take(1)

# take N data from tail
df.tail(1)
```

In [79]:
# convert to pandas DataFrame
df.toPandas()

,a,b,c,d,e
0,1,2.0,string1,2000-01-01,2000-01-01 12:00:00
1,2,3.0,string2,2000-02-01,2000-01-02 12:00:00
2,3,4.0,string3,2000-03-01,2000-01-03 12:00:00


#### Other Examples

```py
# convert to RDD
df.rdd

# convert to JSON
df.toJSON()
```

## Accessing Data

### Data Handling

```py
# Drop duplicates
df.dropDuplicates()

# Replace null values
df.na.fill(50)

# Return new dataframe restricting rows with null values
df.na.drop()

# Return new dataframe replacing one value with another
df.na.replace(10, 20)
```

### Select Column

In [95]:
# select 1 or more columns to show
df.select([df.a, df.c]).show()

+---+-------+
|  a|      c|
+---+-------+
|  1|string1|
|  2|string2|
|  3|string3|
+---+-------+



#### Other Examples

```py
# single column
df.select("a").show(10)

# multiple column
df.select("author", "title", "rank", "price").show(10)

# equivalent of `case-when` in SQL
df.select("title", when(df.title != 'ODD HOURS', 1).otherwise(0)).show(10)

# equivalent of `where x in ...` in SQL
df[df.author.isin("John Sandford", "Emily Giffin")].show(5)

# equivalent of `where x like ...` in SQL
df.select("author", "title", df.title.like("% THE %")).show(15)

# group by
df.groupBy("author").count()

# Returns dataframe column names and data types
df.dtypes

# Displays the content of dataframe
df.show()

# Return first n rows
df.head()

# Returns first row
df.first()

# Return first n rows
df.take(5)

# Computes summary statistics
df.describe().show()

# Returns columns of dataframe
df.columns

# Counts the number of rows in dataframe
df.count()

# Counts the number of distinct rows in dataframe
df.distinct().count()

# Prints plans including physical and logical
df.explain()
```

### Add New Column

In [83]:
from pyspark.sql.functions import upper
df.withColumn('upper_c', upper(df.c)).show()

+---+---+-------+----------+-------------------+-------+
|  a|  b|      c|         d|                  e|upper_c|
+---+---+-------+----------+-------------------+-------+
|  1|2.0|string1|2000-01-01|2000-01-01 12:00:00|STRING1|
|  2|3.0|string2|2000-02-01|2000-01-02 12:00:00|STRING2|
|  3|4.0|string3|2000-03-01|2000-01-03 12:00:00|STRING3|
+---+---+-------+----------+-------------------+-------+



#### Other Examples

```py
# renaming column
df.withColumnRenamed('amazon_product_url', 'URL')

# drop column
df.drop("publisher", "published_date")
# or
df.drop(dataframe.publisher).drop(dataframe.published_date)
```

### Filter data

In [84]:
df.filter(df.a == 1).show()

+---+---+-------+----------+-------------------+
|  a|  b|      c|         d|                  e|
+---+---+-------+----------+-------------------+
|  1|2.0|string1|2000-01-01|2000-01-01 12:00:00|
+---+---+-------+----------+-------------------+



#### Other Examples

```py
df.filter(df["title"] == 'THE HOST').show()
```

### Applying Functions

In [85]:
from pyspark.sql.functions import pandas_udf

# udf -> user-defined function
@pandas_udf('long') # means that this function will return a `long` type
def pandas_plus_one(series: pd.Series) -> pd.Series:
    return series + 1 # add each values with 1

df.select(pandas_plus_one(df.a)).show()

+------------------+
|pandas_plus_one(a)|
+------------------+
|                 2|
|                 3|
|                 4|
+------------------+



### Grouping Data

In [ ]:
df = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])

# grouping and then applying the avg() function to the resulting groups.
df.groupby('color').avg().show()

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
| blue|    3.0|   30.0|
|black|    6.0|   60.0|
+-----+-------+-------+



In [ ]:
def plus_mean(pandas_df):
    return pandas_df.assign(v1=pandas_df.v1 - pandas_df.v1.mean())

# can also apply function
df.groupby('color').applyInPandas(plus_mean, schema=df.schema).show()

## Working with SQL

In [88]:
# create temp table from current DataFrame
df.createOrReplaceTempView("tableA")

# perform select query and show it
spark.sql("SELECT count(*) from tableA").show()

+--------+
|count(1)|
+--------+
|       8|
+--------+



### Using Python Function in SQL

In [92]:
from pyspark.sql.functions import pandas_udf

@pandas_udf("integer")
def add_one(s: pd.Series) -> pd.Series:
    return s + 1

# register udf
spark.udf.register("add_one", add_one)

# apply function in select query
spark.sql("SELECT add_one(v1) FROM tableA").show()

25/01/12 22:40:26 WARN SimpleFunctionRegistry: The function add_one replaced a previously registered function.


+-----------+
|add_one(v1)|
+-----------+
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
+-----------+



## Repartition

**Data Partitioning** refers to the process of dividing a large dataset into smaller chunks or partitions, which can be processed concurrently. 

This is an important aspect of distributed computing, as it allows large datasets to be processed more efficiently by dividing the workload among multiple machines or processors.

```py
# Dataframe with 10 partitions
df.repartition(10).rdd.getNumPartitions()

# Dataframe with 1 partition
df.coalesce(1).rdd.getNumPartitions()
```